In [52]:
import pandas as pd 
import numpy as np


In [53]:
df = pd.read_csv("/home/fwzblk/Projects/prime2/pytorch-prime/p3/my_code/powerplant_data.csv")

In [54]:
df.head()

,AT,V,AP,RH,PE
0,8.34,40.77,1010.84,90.01,480.48
1,23.64,58.49,1011.40,74.20,445.75
2,29.74,56.90,1007.15,41.91,438.76
3,19.07,49.69,1007.22,76.79,453.09
4,11.80,40.66,1017.13,97.20,464.43


In [55]:
#AT represents the temprature
# V represents the vaccum value 
# AP represents the pressure 
# RH represents the humididty 
# PE represnets the produced energy output and is the label for our data 

In [56]:
df.isnull().sum()

AT    0
V     0
AP    0
RH    0
PE    0
dtype: int64

In [57]:
x = df.drop(labels="PE",axis=1)
x

,AT,V,AP,RH
0,8.34,40.77,1010.84,90.01
1,23.64,58.49,1011.40,74.20
2,29.74,56.90,1007.15,41.91
3,19.07,49.69,1007.22,76.79
4,11.80,40.66,1017.13,97.20
...,...,...,...,...
9563,15.12,48.92,1011.80,72.93
9564,33.41,77.95,1010.30,59.72
9565,15.99,43.34,1014.20,78.66
9566,17.65,59.87,1018.58,94.65


In [58]:
y = df['PE']
y

0       480.48
1       445.75
2       438.76
3       453.09
4       464.43
         ...  
9563    462.59
9564    432.90
9565    465.96
9566    450.93
9567    451.67
Name: PE, Length: 9568, dtype: float64

In [59]:
from sklearn.model_selection import train_test_split

x_train , x_test , y_train ,y_test = train_test_split(
    x,y,test_size=0.2,random_state=42
)

In [60]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.fit_transform(x_test)


In [ ]:
import torch 
import torch.nn as nn
x_train_tensor = torch.tensor(x_train_scaled,dtype=torch.float32)
x_test_tensor = torch.tensor(x_test_scaled,dtype=torch.float32)

y_train_tensor = torch.tensor(y_train.values , dtype=float).view(-1,1)
y_test_tensor = torch.tensor(y_test.values , dtype=float).view(-1,1)

In [62]:
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(x_train_tensor,y_train_tensor)

test_dataset = TensorDataset(x_test_tensor,y_test_tensor)

In [63]:
train_loader = DataLoader(train_dataset,batch_size=32,shuffle=True)

test_loader= DataLoader(test_dataset,batch_size=32)

In [64]:
class ANN(nn.Module):
    def __init__(self):
        super(ANN, self).__init__()

        self.model = nn.Sequential(
            # first hidden layer
            nn.Linear(x_train.shape[1], 6),
            nn.ReLU(),

            # second hidden layer
            nn.Linear(6, 6),
            nn.ReLU(),

            # output layer
            nn.Linear(6, 1)
        )

    def forward(self, x):
        return self.model(x)

In [65]:
import torch.optim as optim 

model = ANN()

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters())


In [ ]:
# train the ann
epochs = 100 
train_losses = []
val_losses=[]

for epoch in range(epochs):
    model.train()
    running_loss = 0.0 

    for xb,yb in train_loader:
        optimizer.zero_grad()
        output = model(xb)
        loss = criterion(output,yb)
        loss.backward()
        optimizer.step()

        running_loss+=loss.item()

    epoch_train_loss = running_loss/len(train_loader)
    train_losses.append(epoch_train_loss)

    #validation
    model.eval()
    running_val_loss=0.0

    with torch.no_grad():
        for xb,yb in test_loader:
            outputs = model(xb)
            v_loss = criterion(outputs,yb)
            running_val_loss += v_loss

    epoch_val_loss = running_val_loss/len(test_loader)
    val_losses.append(epoch_val_loss)